In [2]:
import os
os.environ['HF_HOME'] = r'F:\HF_models'

In [ ]:
import torch
import open_clip
from PIL import Image
import numpy as np
import random

# -------------------------------
# Deterministic setup
# -------------------------------
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# -------------------------------
# Load CLIP model and tokenizer
# -------------------------------
model, _, preprocess = open_clip.create_model_and_transforms(
    model_name="ViT-B-32",
    pretrained="openai"   # or "laion2b_s34b_b79k" for broader generalization
)
tokenizer = open_clip.get_tokenizer("ViT-B-32")

# -------------------------------
# Define prompts
# -------------------------------
class_prompts = [
    "a photo of a healthy leaf",
    "a photo of a diseased leaf"
]
text = tokenizer(class_prompts)

# -------------------------------
# Preprocess your input image
# -------------------------------
image = preprocess(Image.open(r"C:\Users\hp5cd\Downloads\diseased.jpg")).unsqueeze(0)

# -------------------------------
# Compute similarities
# -------------------------------
with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    
    similarity = (image_features @ text_features.T)
    probs = similarity.softmax(dim=-1)

# -------------------------------
# Output
# -------------------------------
classes = ["Healthy", "Diseased"]
pred = classes[probs.argmax()]
confidence = probs.max().item()

print(f"Prediction: {pred}, Confidence: {confidence:.4f}")
